In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [ ]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [ ]:
import numpy as np
from lets_plot import LetsPlot
from xaitimesynth import (
    TimeSeriesBuilder,
    random_walk,
    gaussian,
    # sine_wave,
    level_change,
    peak,
)
from xaitimesynth import (
    shapelet,
    manual,
)
from xaitimesynth.visualization import create_ts_visualization

# Enable lets_plot for notebook display
LetsPlot.setup_html()


# Helper functions

In [ ]:
# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location (same position in both)
    .add_feature(
        shapelet(scale=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak only to dimension 2
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    # Add level change to all dimensions, with different locations for each
    # .add_feature(
    #     level_change(amplitude=0.7),
    #     random_location=True,
    #     length_pct=0.2,
    #     dim=[0, 1, 2],
    #     shared_location=False,
    # )
    .build()
)

print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# Example of accessing a specific dimension
dim0_data = multivariate_dataset["X"][:, :, 0]  # First dimension for all samples
print(f"Dimension 0 shape: {dim0_data.shape}")

# Convert to DataFrame with specific dimensions
df = TimeSeriesBuilder().to_df(multivariate_dataset, dimensions=[0, 1])
print(df.head())

create_ts_visualization(multivariate_dataset)


In [ ]:
# Import necessary libraries
from lets_plot import LetsPlot
from xaitimesynth import TimeSeriesBuilder, random_walk, gaussian, shapelet, peak
from xaitimesynth.visualization import create_ts_visualization

# Enable lets_plot for notebook display
LetsPlot.setup_html()

# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location
    .add_feature(
        shapelet(scale=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak to all dimensions
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    .build()
)

# Print dataset information
print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# You can customize the visualization if needed
custom_viz = create_ts_visualization(
    multivariate_dataset,
    components=["aggregated", "features"],
    free_y=False,
    panel_width=300,
    panel_height=200,
    dimensions=None,
)
if len(custom_viz) > 1:
    for viz in custom_viz:
        viz.show()
else:
    custom_viz.show()


In [ ]:
# For a single plot or list with one element
viz = create_ts_visualization(multivariate_dataset, dimensions=[0])
# viz.show()

# For multiple plots
vizs = create_ts_visualization(multivariate_dataset, dimensions=[0, 1])
for i, viz in enumerate(vizs):
    print(f"Dimension {i}:")
    viz.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def visualize_multivariate_ts(dataset, sample_idx=0, class_idx=None, figsize=(12, 8)):
    """Visualize a multivariate time series sample with feature overlays.

    Args:
        dataset: Dictionary containing dataset from TimeSeriesBuilder.
        sample_idx: Index of the sample to visualize.
        class_idx: If provided, choose the first sample from this class.
        figsize: Figure size tuple.
    """
    # If class_idx is provided, find the first sample of that class
    if class_idx is not None:
        class_samples = np.where(dataset["y"] == class_idx)[0]
        if len(class_samples) == 0:
            print(f"No samples found for class {class_idx}")
            return
        sample_idx = class_samples[0]

    # Get data for the sample
    X_sample = dataset["X"][sample_idx]
    class_label = dataset["y"][sample_idx]
    n_dims = X_sample.shape[1]

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Display as heatmap
    im = ax.imshow(X_sample.T, cmap="gray_r", aspect="auto", interpolation="nearest")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Value")

    # Add feature overlays from feature masks
    masks_added = 0
    for key, mask_array in dataset["feature_masks"].items():
        if key.startswith(f"class_{class_label}_") and sample_idx < mask_array.shape[0]:
            mask = mask_array[sample_idx]

            # Extract dimension from feature name
            if "_dim" in key:
                feature_parts = key.split("_dim")
                dim_idx = int(feature_parts[-1])

                # Find contiguous blocks of True values
                changes = np.diff(np.concatenate(([False], mask, [False])))
                starts = np.where(changes == True)[0]
                ends = np.where(changes == False)[0]

                for start, end in zip(starts, ends):
                    # Draw rectangle for this feature region
                    rect = patches.Rectangle(
                        (start, dim_idx - 0.5),
                        end - start,
                        1,
                        linewidth=1,
                        edgecolor="r",
                        facecolor="r",
                        alpha=0.3,
                    )
                    ax.add_patch(rect)
                    masks_added += 1

    # Set labels and title
    ax.set_title(f"Sample {sample_idx} (Class {class_label})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Dimension")

    # Set y-axis ticks in middle of each row
    ax.set_yticks(np.arange(n_dims))
    ax.set_yticklabels([f"Dim {i}" for i in range(n_dims)])

    plt.tight_layout()
    print(f"Added {masks_added} feature overlays")
    plt.show()


# Example usage:
# For class 0 (no features)
visualize_multivariate_ts(multivariate_dataset, class_idx=0)

# For class 1 (with features)
visualize_multivariate_ts(multivariate_dataset, class_idx=1)

# To visualize a specific sample
# visualize_multivariate_ts(multivariate_dataset, sample_idx=10)

# Synth Data

In [ ]:
from xaitimesynth import (
    list_components,
    list_signal_components,
    list_feature_components,
)

print(list_components())
print(list_signal_components())
print(list_feature_components())


In [ ]:
"""
Example usage of the XAITimeSynth package.

This example demonstrates how to create a synthetic time series dataset
with two classes, where one has discriminative features and the other doesn't.
"""


# Create a dataset with two classes using the fluent API
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), start_pct=0.4, end_pct=0.6)
    .add_feature(level_change(amplitude=0.5), start_pct=0.7, end_pct=0.9)
    .build(train_test_split=0.8)
)

# Print dataset structure
print("Dataset keys:", list(dataset.keys()))
print(f"X shape: {dataset['X'].shape}")
print(f"y shape: {dataset['y'].shape}")
print(f"Classes: {np.unique(dataset['y'])}")
print(f"Class distribution: {np.bincount(dataset['y'])}")


# Example with random feature locations
random_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features with random locations
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), random_location=True, length_pct=0.2)
    .add_feature(level_change(amplitude=0.5), random_location=True, length_pct=0.2)
    .build()
)


# Create a custom component and use it
def my_seasonal_pattern(n_timesteps, rng, frequency=0.1, amplitude=1.0):
    """Generate a custom seasonal pattern."""
    t = np.arange(n_timesteps)
    return amplitude * np.sin(2 * np.pi * frequency * t / n_timesteps)


custom_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=10)
    .for_class(0)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)


In [ ]:
# Create a dataset with the specified characteristics and higher frequency sine wave
sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(
        0
    )  # Class 0: Higher frequency sine wave pattern as base signal and level shift
    .add_signal(
        manual(
            generator=lambda n_timesteps, rng: np.sin(
                5 * np.linspace(0, 2 * np.pi, n_timesteps)
            )
        )
    )  # 5 cycles
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)  # Class 1: Constant zero and level shift
    .add_signal(manual(generator=lambda n_timesteps, rng: np.zeros(n_timesteps)))
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)


custom_order_plot = create_ts_visualization(sine_levelshift_dataset)
custom_order_plot.show()

In [ ]:
random_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, random_state=42)
    # Class 0: No discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk with randomly located features
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.2), random_location=True, length_pct=0.15)
    .add_feature(peak(amplitude=1.0, width=5), random_location=True, length_pct=0.1)
    .build()
)
print(random_dataset.keys())


custom_order_plot = create_ts_visualization(random_dataset)
custom_order_plot.show()

In [ ]:
# Build a dataset
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_feature(shapelet(scale=1.0), start_pct=0.4, end_pct=0.6)
    .build()
)

# Convert to pandas DataFrame
builder = TimeSeriesBuilder()
df = builder.to_df(dataset)

# # Now use pandas operations for analysis or custom plotting
# import matplotlib.pyplot as plt
# pivot_df = df.pivot_table(index='time', columns=['class', 'component'], values='value')
# pivot_df.plot(figsize=(12, 6))
# plt.show()
custom_order_plot = create_ts_visualization(dataset)
custom_order_plot.show()

In [ ]:
# import numpy as np
# from lets_plot import *
# from xaitimesynth import (
#     TimeSeriesBuilder,
#     random_walk,
#     gaussian,
#     # sine_wave,
#     level_change,
#     peak,
# )
from xaitimesynth.visualization import (
    plot_class_comparison,
)


# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(
        n_timesteps=100,
        n_samples=30,
        random_state=42,
    )
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)


# Example 1: Default visualization with ordered components
# Note: Default order is now "Full Series", "Features", "Base Structure", "Noise"
print("Default visualization with ordered components:")
default_plot = create_ts_visualization(dataset)
default_plot.show()


# Example 5: Class comparison alternatives
print("\nClass comparison - default (one panel per class):")
default_comparison = plot_class_comparison(dataset, panel_width=1000, panel_height=150)
default_comparison.show()


In [ ]:
# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=30, random_state=42)
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # .add_feature(sine_wave(frequency=0.2, amplitude=1.5), start_pct=0.3, end_pct=0.5)
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)
print(dataset.keys())


In [ ]:
import numpy as np


import numpy as np


import numpy as np

from xaitimesynth import plot_signal
from xaitimesynth.generators import generate_random_walk
import numpy as np

signal = generate_random_walk(100, np.random.RandomState(42), step_size=0.2)
plot_signal(signal)


# Generate and plot a random walk
plot_signal(component_type="random_walk", n_timesteps=100, step_size=0.2).show()

# Generate and plot a seasonal pattern
plot_signal(component_type="seasonal", n_timesteps=200, period=20, amplitude=1.5).show()

# Example 1: Using a manual component with custom values

# Create custom values for a zigzag pattern
zigzag = np.concatenate([np.linspace(0, 1, 10), np.linspace(1, 0, 10)])
zigzag = np.tile(zigzag, 5)  # Repeat the pattern 5 times

# Plot using the manual component with custom values
plot_signal(
    component_type="manual",
    values=zigzag,
    n_timesteps=100,
    line_color="red",
).show()


# Define a custom generator function for a damped oscillation
def damped_oscillation(length, rng, frequency=0.1, decay=0.05, **kwargs):
    t = np.arange(length)
    return np.exp(-decay * t) * np.sin(2 * np.pi * frequency * t)


# Plot using the manual component with custom generator
plot_signal(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
).show()

plot_signal(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
    normalization_kwargs={"feature_range": (-1, 1)},  # Custom normalization range
).show()


In [ ]:
from xaitimesynth.generators import generate_ecg_like

plot_signal(generate_ecg_like(400, np.random.RandomState(42))).show()
plot_signal(component_type="ecg_like", n_timesteps=400, line_color="blue").show()

In [ ]:
import numpy as np


import numpy as np


# Normal adult ECG at rest
plot_signal(
    component_type="ecg_like",
    n_timesteps=1000,
    heart_rate=65,
    line_color="blue",
    width=700,
    height=350,
).show()

# Tachycardia (elevated heart rate)
plot_signal(
    component_type="ecg_like",
    n_timesteps=1000,
    heart_rate=120,
    line_color="red",
    width=700,
    height=350,
).show()

# Bradycardia (slow heart rate)
plot_signal(
    component_type="ecg_like",
    n_timesteps=1000,
    heart_rate=45,
    line_color="orange",
    width=700,
    height=350,
).show()
